# 06 -- Expected Loss (EL = PD x LGD x EAD)

**What this notebook does (plain English):** Brings the three pieces together.
**Expected Loss** is the average loss a lender should budget for:

> **Expected Loss = chance of default (PD) x loss if it defaults (LGD) x amount
> owed (EAD)**

We score every loan, total it into a portfolio number, and sort loans into the
accounting **IFRS 9 / AASB 9 stages** (1 = healthy, 2 = deteriorating, 3 =
defaulted). We also walk through the full sum for one example loan.

**Headline result:** a single portfolio Expected Loss figure, dominated by the
Stage 3 (already-defaulted) loans and by the crisis vintages.

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the base table and build the three components for EVERY loan.
import pandas as pd
import numpy as np
from src import models
from src.output import save_csv
base = pd.read_parquet('data/processed/analysis_base.parquet').copy()

In [3]:
# PD for every loan (logistic model fit on the whole book).
pd_model, pd_cols = models.fit_pd(base)
base['pd_hat'] = models.predict_pd(pd_model, pd_cols, base)

In [4]:
# LGD for every loan (two-stage model trained on disposed defaults).
disposed = base[base['disposed'] & base['lgd'].notna()]
lgd_model = models.TwoStageLGD().fit(disposed)
base['lgd_hat'] = lgd_model.predict(base)

In [5]:
# EAD for every loan: balance at default if it defaulted, else the original
# loan amount as the exposure proxy for a still-performing loan.
base['ead_loan'] = np.where(base['ever_default'], base['ead'], base['original_upb'])
# Expected loss per loan = PD x LGD x EAD.
base['expected_loss'] = base['pd_hat'] * base['lgd_hat'] * base['ead_loan']

In [6]:
# IFRS 9 / AASB 9 staging: 3 = defaulted (credit-impaired), 2 = significant
# increase in risk (ever 60+ days late but not defaulted), 1 = performing.
stage2 = (~base['ever_default']) & (base['max_delinq_status'].fillna(0) >= 2)
base['ifrs9_stage'] = np.where(base['ever_default'], 3, np.where(stage2, 2, 1))
# Stage 1 carries a 12-month EL; Stages 2 & 3 carry lifetime EL (here our PD is
# a lifetime/observed PD, so we show a 12-month view as one quarter of it).
base['el_reported'] = np.where(base['ifrs9_stage'] == 1, base['expected_loss'] * 0.25, base['expected_loss'])

In [7]:
# Portfolio Expected Loss summary by IFRS 9 stage (the saved result).
el_summary = base.groupby('ifrs9_stage').agg(
    loans=('loan_sequence_number', 'size'),
    avg_pd=('pd_hat', 'mean'),
    avg_lgd=('lgd_hat', 'mean'),
    total_ead=('ead_loan', 'sum'),
    lifetime_expected_loss=('expected_loss', 'sum'),
    reported_expected_loss=('el_reported', 'sum'),
).reset_index().round(2)
save_csv(el_summary, 'output/06_expected_loss.csv')
el_summary

,ifrs9_stage,loans,avg_pd,avg_lgd,total_ead,lifetime_expected_loss,reported_expected_loss
0,1,132177,0.06,0.44,2.703702e+10,8.008088e+08,2.002022e+08
1,2,6067,0.14,0.50,1.140920e+09,8.169057e+07,8.169057e+07
2,3,11756,0.20,0.53,2.247190e+09,2.283437e+08,2.283437e+08


In [8]:
# Worked example: show PD x LGD x EAD = EL for a single representative loan.
ex = base.sort_values('expected_loss', ascending=False).iloc[100]
print('Worked example loan:', ex['loan_sequence_number'])
print(f"  PD  (chance of default) = {ex['pd_hat']:.3f}")
print(f"  LGD (loss if default)   = {ex['lgd_hat']:.3f}")
print(f"  EAD (amount owed)       = ${ex['ead_loan']:,.0f}")
print(f"  Expected Loss = {ex['pd_hat']:.3f} x {ex['lgd_hat']:.3f} x ${ex['ead_loan']:,.0f} = ${ex['expected_loss']:,.0f}")

Worked example loan: F07Q30149213
  PD  (chance of default) = 0.589
  LGD (loss if default)   = 0.459
  EAD (amount owed)       = $317,256
  Expected Loss = 0.589 x 0.459 x $317,256 = $85,696


**Reading the table:** Stage 3 holds the already-defaulted loans and
carries most of the loss; Stage 1 is the large healthy book on a 12-month view.
The worked example shows the headline equation end-to-end for one loan.